📊 Día 02: SQL Avanzado con DuckDB - Subqueries y Lógica de Negocio
Fecha: 2026-03-03
Objetivo: Implementar subconsultas para resolver preguntas complejas de negocio y configurar la reproducibilidad del entorno.

--------------------------------------------------------------------------------
🛠 1. Setup y Reproducibilidad
Siguiendo la Regla 10, este notebook utiliza una conexión persistente a DuckDB para asegurar que los resultados sean consistentes.
import duckdb
import pandas as pd

# Conexión a la base de datos en la carpeta profesional /data
con = duckdb.connect('../data/analytics.db')
print("DuckDB conectado exitosamente.")

In [1]:
import duckdb
import pandas as pd

In [2]:
# Conectarse a DuckDB
con = duckdb.connect('../data/analytics.db')

con.execute("SHOW TABLES").fetchdf()


,name
0,customers
1,transactions


In [ ]:
# imrpimir las tablas
con.execute("SELECT * FROM transactions").fetchdf()

,transaction_id,customer_id,amount,date
0,101,1,300.0,2024-01-25
1,102,1,100.0,2024-01-28
2,103,2,150.0,2024-01-10
3,104,2,80.0,2024-01-15
4,105,3,200.0,2024-01-11
5,106,3,50.0,2024-01-22
6,107,4,75.0,2024-02-01
7,108,5,500.0,2024-02-05
8,109,6,250.0,2024-02-08
9,110,7,300.0,2024-02-10


In [5]:
con.execute("SELECT * FROM customers").fetchdf()


,customer_id,name,city
0,1,Ana,Bogotá
1,2,Luis,Medellín
2,3,Pedro,Cali
3,4,Sofía,Barranquilla
4,5,Carlos,Bogotá
5,6,Laura,Medellín
6,7,María,Cali
7,8,Andrés,Bogotá


🔹 1️⃣ Subquery en WHERE
🎯 Problema:

Clientes que gastaron más que el promedio general.

In [ ]:
con.execute("""
SELECT customer_id, sum(amount) as total_amount
FROM transactions
GROUP BY customer_id
HAVING sum(amount) > (select avg(amount) from transactions)
order by total_amount desc
""").fetchdf()

,customer_id,total_amount
0,5,500.0
1,1,400.0
2,7,300.0
3,3,250.0
4,6,250.0
5,2,230.0


🔹 2️⃣ Subquery con tabla derivada (subquery en FROM)
🎯 Problema:

Clientes que hicieron más transacciones que el promedio de clientes.

In [21]:
con.execute("""
SELECT customer_id, num_transactions
FROM (
    -- Paso 1: Número de transacciones por cliente
    SELECT customer_id, COUNT(*) AS num_transactions
    FROM transactions
    GROUP BY customer_id
) AS t
-- Paso 3: Filtrar los que superan el promedio
WHERE num_transactions > (
    -- Paso 2: Promedio de número de transacciones por cliente
    SELECT AVG(num_transactions)
    FROM (
        SELECT customer_id, COUNT(*) AS num_transactions
        FROM transactions
        GROUP BY customer_id
    ) AS t_avg
)
ORDER BY num_transactions DESC;
""").fetch_df()

,customer_id,num_transactions
0,1,2
1,2,2
2,3,2


🔹 Regla rápida

Si estás dentro de un GROUP BY y filtras sobre una agregación → HAVING

Si estás fuera del GROUP BY y solo comparas columnas existentes con un valor → WHERE

🔹 3️⃣ Subquery correlacionada (nivel más alto)

Ahora subimos el nivel.

🎯 Problema:

clientes con gasto total mayor que el promedio de gasto total por cliente

In [26]:
con.execute("""
            SELECT customer_id, total_gasto
FROM (
    SELECT customer_id, SUM(amount) AS total_gasto
    FROM transactions
    GROUP BY customer_id
) AS gtc
WHERE total_gasto > (
    SELECT AVG(total_gasto)
    FROM (
        SELECT customer_id, SUM(amount) AS total_gasto
        FROM transactions
        GROUP BY customer_id
    ) AS t_avg
)
ORDER BY total_gasto DESC;SELECT customer_id, sum(amount) from transactions
            GROUP BY customer_id
            """).fetchdf()

,customer_id,sum(amount)
0,1,400.0
1,2,230.0
2,3,250.0
3,4,75.0
4,5,500.0
5,6,250.0
6,7,300.0
7,8,50.0
